# imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", None)

In [2]:
csv_path = Path(r"C:\Users\user\Desktop\div\data\files\housing.csv")
df = pd.read_csv(csv_path)
df.head(3)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY


# outliers

### simple

In [3]:
outliers1 = df['median_house_value'] == df['median_house_value'].max()
outliers2 = df['housing_median_age'] == df['housing_median_age'].max()
outliers = outliers1 | outliers2


In [4]:
df = df.loc[~outliers, :].copy()
print(df.shape)
df.head(3)

(18572, 10)


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
8,-122.26,37.84,42.0,2555.0,665.0,1206.0,595.0,2.0804,226700.0,NEAR BAY


# split

In [5]:
X = df.drop('median_house_value', axis='columns')
y = df['median_house_value']

In [6]:
X['income_cat'] = pd.cut(
    X['median_income'],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
    )

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=X['income_cat'],
    test_size=0.2
    )

X_train.shape, X_test.shape

((14857, 10), (3715, 10))

In [8]:
X_train.drop('income_cat', axis='columns', inplace=True)
X_test.drop('income_cat', axis='columns', inplace=True)

# pipeline

In [9]:
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.cluster import KMeans

n_clusters = 10
kmeans_ = KMeans(n_clusters = n_clusters)
kmeans_.fit(X_train[['latitude', 'longitude']])
print(kmeans_.cluster_centers_)

[[  34.00470809 -118.21610099]
 [  37.26411284 -121.93094284]
 [  40.49615591 -123.0236828 ]
 [  32.90480548 -116.94577089]
 [  36.32808989 -119.44871297]
 [  34.01555388 -117.34829691]
 [  38.0052749  -122.32053348]
 [  37.76934447 -120.95140103]
 [  38.92251996 -121.27345164]
 [  34.8036     -120.13250667]]


In [10]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans

class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10):
        self.n_clusters = n_clusters

    def fit(self, X, y=None):
        self.kmeans_ = KMeans(self.n_clusters)
        self.kmeans_.fit(X)
        return self  

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=1).round(3)

    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

cat_cols = ['ocean_proximity']
log_cols = [
    'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income'
    ]

cat_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])
log_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('log', FunctionTransformer(np.log1p, inverse_func=np.expm1, feature_names_out='one-to-one')),
    ('scl', StandardScaler())
])

def ratio_func(arr_2d):
    return arr_2d[:, [0]] / arr_2d[:, [1]]
def ratio_feature_names(*args, **kwargs):
    return ['ratio']

rat_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('newcol', FunctionTransformer(ratio_func, feature_names_out=ratio_feature_names)),
    ('scl', StandardScaler())
])
cluster_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('cluster', ClusterSimilarity(n_clusters=10))
])
num_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('scl', StandardScaler())
])

preprocessing = ColumnTransformer(
    transformers=[
        ('CAT', cat_pipeline, cat_cols),
        ('LOG', log_pipeline, log_cols),
        ('RAT_b/r', rat_pipeline, ['total_bedrooms', 'total_rooms']),
        ('RAT_r/h', rat_pipeline, ['total_rooms', 'households']),
        ('RAT_p/h', rat_pipeline, ['population', 'households']),
        ("GEO", cluster_pipeline, ["latitude", "longitude"]),
    ], remainder=num_pipeline
)

X_train_arr = preprocessing.fit_transform(X_train)
X_train_prepared = pd.DataFrame(X_train_arr, columns=preprocessing.get_feature_names_out())

X_test_arr = preprocessing.fit_transform(X_test)
X_test_prepared = pd.DataFrame(X_test_arr, columns=preprocessing.get_feature_names_out())

X_train_prepared

,CAT__ocean_proximity_<1H OCEAN,CAT__ocean_proximity_INLAND,CAT__ocean_proximity_ISLAND,CAT__ocean_proximity_NEAR BAY,CAT__ocean_proximity_NEAR OCEAN,LOG__total_rooms,LOG__total_bedrooms,LOG__population,LOG__households,LOG__median_income,RAT_b/r__ratio,RAT_r/h__ratio,RAT_p/h__ratio,GEO__Cluster 0 similarity,GEO__Cluster 1 similarity,GEO__Cluster 2 similarity,GEO__Cluster 3 similarity,GEO__Cluster 4 similarity,GEO__Cluster 5 similarity,GEO__Cluster 6 similarity,GEO__Cluster 7 similarity,GEO__Cluster 8 similarity,GEO__Cluster 9 similarity,remainder__housing_median_age
0,1.0,0.0,0.0,0.0,0.0,0.124221,-0.646876,-0.451939,-0.605692,2.067540,-1.605288,1.393242,0.008988,0.000,0.498,0.002,0.006,0.000,0.000,0.367,0.000,0.000,0.000,-0.084802
1,0.0,1.0,0.0,0.0,0.0,0.380382,0.524040,0.223864,0.395821,-0.488094,0.236084,-0.104858,-0.050188,0.000,0.990,0.000,0.094,0.000,0.000,0.045,0.000,0.000,0.000,0.265464
2,1.0,0.0,0.0,0.0,0.0,0.665390,0.571903,0.882252,0.643426,0.797551,-0.387684,-0.034558,0.030756,0.000,0.377,0.003,0.003,0.000,0.000,0.466,0.000,0.000,0.000,-0.259936
3,1.0,0.0,0.0,0.0,0.0,-1.614895,-0.657096,-0.023041,-0.605692,-1.336277,3.618057,-1.235514,0.115056,0.000,0.960,0.000,0.058,0.000,0.000,0.082,0.000,0.000,0.000,1.141131
4,0.0,0.0,0.0,1.0,0.0,-1.227000,-1.078571,-1.807681,-0.951976,-0.817892,0.398034,-0.521009,-0.142248,0.852,0.000,0.001,0.000,0.001,0.298,0.000,0.141,0.821,0.000,0.878431
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14852,0.0,0.0,0.0,0.0,1.0,-0.048896,-0.010783,-0.063105,0.084981,0.312664,-0.007203,-0.295282,-0.046792,0.749,0.000,0.000,0.000,0.000,0.188,0.000,0.063,0.870,0.000,0.790864
14853,0.0,1.0,0.0,0.0,0.0,-0.388930,-0.265702,-0.443794,-0.289984,-1.293230,0.250708,-0.258432,-0.048511,0.001,0.000,0.000,0.000,0.187,0.000,0.000,0.001,0.000,0.738,0.002764
14854,0.0,1.0,0.0,0.0,0.0,1.222183,0.984176,-0.881771,-0.513622,-0.069150,-0.744286,5.401695,-0.082430,0.002,0.000,0.000,0.000,0.530,0.002,0.000,0.121,0.000,0.001,-1.135602
14855,1.0,0.0,0.0,0.0,0.0,-0.262206,-0.483622,-0.312235,-0.363583,0.325564,-0.604769,0.054496,-0.011289,0.000,0.998,0.000,0.092,0.000,0.000,0.050,0.000,0.000,0.000,1.141131


# training

In [17]:
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import LinearRegression

full_pipeline = Pipeline([
    ('cleaning', preprocessing),
    ('lin_reg', LinearRegression()),
])

full_pipeline.fit(X_train, y_train)
y_pred = full_pipeline.predict(X_test)
# y_test.to_numpy()
root_mean_squared_error(y_test.to_numpy(), y_pred)

59977.26816144859

In [24]:
from sklearn.ensemble import RandomForestRegressor

full_pipeline_rf = Pipeline([
    ('cleaning', preprocessing),
    ('rf_reg', RandomForestRegressor()),
])

full_pipeline_rf.fit(X_train, y_train)
y_pred = full_pipeline_rf.predict(X_test)
# y_test.to_numpy()
root_mean_squared_error(y_test.to_numpy(), y_pred)

43909.32035512756

In [25]:
from sklearn.ensemble import BaggingRegressor

full_pipeline_bag = Pipeline([
    ('cleaning', preprocessing),
    ('bag_reg', BaggingRegressor(RandomForestRegressor(),)),
])

full_pipeline_bag.fit(X_train, y_train)
y_pred = full_pipeline_bag.predict(X_test)
# y_test.to_numpy()
root_mean_squared_error(y_test.to_numpy(), y_pred)

44220.60047823395

In [23]:
# from sklearn.ensemble import VotingRegressor


# full_pipeline = Pipeline([
#     ('cleaning', preprocessing),
#     ('vot_reg', VotingRegressor([
#         ('lin_reg', LinearRegression()),
#         ('rf_reg', RandomForestRegressor()),
#         ])
#     ),
# ])

# full_pipeline.fit(X_train, y_train)
# y_pred = full_pipeline.predict(X_test)
# # y_test.to_numpy()
# root_mean_squared_error(y_test.to_numpy(), y_pred)

In [28]:
from sklearn.model_selection import cross_val_score

cross_val_score(
    full_pipeline_rf, 
    X.drop('income_cat', axis='columns'), 
    y, 
    cv=10, 
    scoring='neg_root_mean_squared_error'
)

array([-44228.33266731, -42474.76163266, -44420.27168925, -42939.63159967,
       -39892.11143308, -40020.36631921, -41139.36265652, -40626.23373874,
       -44168.78453598, -41928.85487141])

# Tuning

In [30]:
full_pipeline_rf.named_steps

{'cleaning': ColumnTransformer(remainder=Pipeline(steps=[('imp',
                                              SimpleImputer(strategy='median')),
                                             ('scl', StandardScaler())]),
                   transformers=[('CAT',
                                  Pipeline(steps=[('imp',
                                                   SimpleImputer(strategy='most_frequent')),
                                                  ('ohe',
                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                  ['ocean_proximity']),
                                 ('LOG',
                                  Pipeline(steps=[('imp',
                                                   SimpleImputer(strategy='median')),
                                                  ('log',
                                                   FunctionTran...
                                                   SimpleImputer(strat

In [ ]:
RandomForestRegressor()

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

param_grid = {
    # 'cleaning__GEO__cluster__n_clusters' : [5, 10, 15],
    'rf_reg__n_estimators' : [50, 100, 150], 
    # 'cleaning__LOG__imp__strategy' : ['median', 'mean'] 
}

grid_search = GridSearchCV(
    full_pipeline_rf, 
    param_grid = param_grid, 
    cv=3, 
    scoring='neg_root_mean_squared_error'
)
# RandomizedSearchCV
grid_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...Regressor())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'cleaning__GEO__cluster__n_clusters': [5, 10, ...], 'cleaning__LOG__imp__strategy': ['median', 'mean'], 'rf_reg__n_estimators': [50, 100, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : 

In [36]:
grid_search.best_params_

{'cleaning__GEO__cluster__n_clusters': 15,
 'cleaning__LOG__imp__strategy': 'mean',
 'rf_reg__n_estimators': 100}

In [38]:
import pandas as pd

In [43]:
cv_res = pd.DataFrame(grid_search.cv_results_)
cv_res.loc[:, [
       'param_cleaning__GEO__cluster__n_clusters',
       'param_cleaning__LOG__imp__strategy', 
       'param_rf_reg__n_estimators',
       'split0_test_score', 'split1_test_score', 'split2_test_score',
       'mean_test_score', 'rank_test_score']].sort_values(by='rank_test_score')

,param_cleaning__GEO__cluster__n_clusters,param_cleaning__LOG__imp__strategy,param_rf_reg__n_estimators,split0_test_score,split1_test_score,split2_test_score,mean_test_score,rank_test_score
16,15,mean,100,-43717.137769,-43214.701001,-42559.118125,-43163.652298,1
14,15,median,150,-43848.249809,-43046.321425,-42658.365909,-43184.312381,2
13,15,median,100,-43582.220323,-43362.490728,-42676.398470,-43207.036507,3
17,15,mean,150,-43664.294607,-43324.663192,-42852.915365,-43280.624388,4
15,15,mean,50,-44083.988368,-43543.507919,-42696.302006,-43441.266098,5
8,10,median,150,-43899.789033,-43664.205753,-43053.225711,-43539.073499,6
10,10,mean,100,-44095.113786,-43625.055639,-43055.756187,-43591.975204,7
12,15,median,50,-44157.969387,-43676.953296,-43059.362894,-43631.428526,8
11,10,mean,150,-43932.121523,-43754.659027,-43472.269793,-43719.683448,9
7,10,median,100,-44164.467176,-44020.058306,-43335.261606,-43839.929029,10
